# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates a full workflow for exploring, processing, and visualizing a dataset defined by a Croissant schema using the `mlcroissant` library. All dataset entities (record sets, fields, columns) are referenced by their `@id`. The approach follows best practices for FAIR data access and reproducible EDA.

### Dataset Source

Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata
metadata = dataset.metadata
print(metadata.name)
print(metadata.description)


## 2. Data Overview

Review available record sets and fields, referencing their `@id` values.

In [ ]:
# List all record sets defined in the dataset
record_sets = dataset.metadata.record_set
if record_sets:
    print('Record Sets available:')
    for rs in record_sets:
        print(f"  - Name: {getattr(rs, 'name', '')}, @id: {rs['@id'] if isinstance(rs, dict) else rs._id}")
        # List fields/columns for each record set
        if hasattr(rs, 'field') and rs.field:
            print('    Fields:')
            for field in rs.field:
                print(f"      - Name: {getattr(field, 'name', '')}, @id: {field['@id'] if isinstance(field, dict) else field._id}")
else:
    print("No record sets found in metadata. Trying to infer from records...")

# Since the schema may be complex, use dataset.records() to infer available record_set IDs
available_record_set_ids = dataset.discover_record_sets()
print('Record sets discovered by mlcroissant:')
for rs_id in available_record_set_ids:
    print(f'  - {rs_id}')
    # Preview a few records from each
    records = list(dataset.records(record_set=rs_id))
    if records:
        print(f"    Example fields: {list(records[0].keys())}")
    else:
        print("    No records found.")


## 3. Data Extraction

Load data from each discovered record set into a Pandas DataFrame. Use `@id` values to reference record sets and their fields. This enables further analysis and processing.

In [ ]:
# Extract available record sets (by @id)
record_set_ids = available_record_set_ids
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:  # Only load non-empty record sets
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set @id: {record_set_id}")
        print(f"  Columns: {df.columns.tolist()}")

# Choose the main record set for EDA: Use the largest DataFrame (most rows, most columns)
main_record_set_id = max(dataframes, key=lambda k: dataframes[k].shape[0]) if dataframes else None
if main_record_set_id:
    print(f"Main record set selected for EDA: {main_record_set_id}")
    print(f"Preview data:")
    display(dataframes[main_record_set_id].head())
else:
    print("No tabular record sets found.")


## 4. Exploratory Data Analysis (EDA)

Process the data: filtering, normalization, grouping, and basic statistics. All references are via field/column `@id` values.

- Filtering for age > 40 (if column exists)
- Normalizing numeric 'age' field
- Grouping by 'sex' or 'MSI status' (if available)


In [ ]:
# Sample EDA: Choose relevant columns by @id or column name
df = dataframes[main_record_set_id]

# Try to find candidate numeric and group fields
candidate_numeric_fields = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower()]
candidate_group_fields = [col for col in df.columns if 'sex' in col.lower() or 'msi' in col.lower() or 'location' in col.lower()]
print(f"Numeric fields detected: {candidate_numeric_fields}")
print(f"Group-by fields detected: {candidate_group_fields}")

# For demonstration, choose first available numeric and group field
if candidate_numeric_fields:
    numeric_field = candidate_numeric_fields[0]
else:
    numeric_field = df.select_dtypes(include=np.number).columns[0] if len(df.select_dtypes(include=np.number).columns)>0 else df.columns[0]

if candidate_group_fields:
    group_field = candidate_group_fields[0]
else:
    group_field = None

# Filter numeric_field for values greater than threshold
threshold = 40  # Assuming age, interval etc.
if numeric_field in df.columns:
    filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce') > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalization
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (pd.to_numeric(filtered_df[numeric_field], errors='coerce') - pd.to_numeric(filtered_df[numeric_field], errors='coerce').mean()) / pd.to_numeric(filtered_df[numeric_field], errors='coerce').std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Group by group_field and compute mean if possible
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())
else:
    print(f"Numeric field {numeric_field} not in columns. Skipping filter and normalization.")


## 5. Visualization

Visualize distributions and relationships between key fields using matplotlib.

In [ ]:
if numeric_field in df.columns:
    plt.figure(figsize=(8, 5))
    pd.to_numeric(df[numeric_field], errors='coerce').hist(bins=15, color='skyblue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

if group_field and group_field in df.columns:
    plt.figure(figsize=(8, 5))
    df.groupby(group_field)[numeric_field].mean().plot(kind='bar', color='salmon')
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.tight_layout()
    plt.show()


## 6. Conclusion

This notebook demonstrated loading and processing a clinical oncology dataset using `mlcroissant`. Key fields were referenced via `@id`, and common EDA steps were applied: filtering, normalization, grouping, and basic visualization. For more advanced analysis, further exploration of record set, field, and column `@id`s is recommended, as well as connecting metadata to domain context via FAIR principles.